In [1]:
# Decoding 

# Pool data across session and animals
import argparse
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.io as sio
import seaborn as sns
import pickle
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR, LinearSVC
from sklearn.metrics import (
    accuracy_score,
    silhouette_score,
    adjusted_rand_score,
    silhouette_samples,
    confusion_matrix,
)
from sklearn.cluster import AgglomerativeClustering, SpectralClustering, KMeans
from sklearn.model_selection import KFold, LeaveOneOut, train_test_split,  cross_val_score, cross_val_predict
from sklearn.model_selection import GridSearchCV
from sklearn.kernel_ridge import KernelRidge
from sklearn import linear_model
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.ensemble import RandomForestClassifier

from patsy import (
    ModelDesc,
    EvalEnvironment,
    Term,
    EvalFactor,
    LookupFactor,
    dmatrices,
    INTERCEPT,
)
from statsmodels.distributions.empirical_distribution import ECDF
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.colorbar as colorbar
import sys
import utils_py3 as ut

from s2p_utils.data_loader import DataLoader
from s2p_utils.processing_utils import (
    extract_events,
    get_corrected_F,
    normalize_signal,
    extract_Fave_around_events,
    extract_F_around_events,
    reorder_clusters,
    filter_trials_by_minITI,
    extract_patterns_for_decoding,
    get_animal_decoding_dict,
    do_time_resolved_decoding
)
from plot_utils import (
    plot_decoding_accuracy_across_time
)

logger = logging.getLogger(__name__)

In [ ]:
# 1. Initialize parameters
framerate = 5
trial_types = ["CS1+", "CS2+", "CS3-"]
pre_cue_window = 3
post_cue_window = 17
delay_to_reward = 3
# min_cell_prob = 0.5
# neucoeff = 0.7
# cell_threshold = 10
data_dir = "Z:\\2p\\experiment1"

imaging_system = "INSS"
learning_stage = "late"

# Set animals and days for early and late learning
if learning_stage == "early":
    result_dir = "Z:\\2p\\experiment1\\population_data\\early learning\\d1_first10_trials\\"
    animal_list = [
        "MZ_CA1_WD_F3",
        "MZ_CA1_WD_M4",
        "MZ_CA1_WD_M5",
        "MZ_CA1_WD_M6",
        "MZ_CA1_WD_M7",
        "MZ_CA1_WD_M8",
        "MZ_CA1_WD_JB_54",
        "MZ_CA1_WD_JB_55",
    ]
    daylist = [1, 1, 1, 1, 1, 1, 1, 1]
    subtrials = 'first10'

elif learning_stage == "late":
    result_dir = "Z:\\2p\\experiment1\\population_data\\late learning\\all trials"
    animal_list = [
        "MZ_CA1_WD_F3",
        "MZ_CA1_WD_M4",
        "MZ_CA1_WD_M5",
        "MZ_CA1_WD_M6",
        "MZ_CA1_WD_M7",
        "MZ_CA1_WD_M8",
        "MZ_CA1_WD_JB_54",
        "MZ_CA1_WD_JB_55"
    ]   
    # daylist = [7, 5, 6, 6, 5, 6, 8, 12] # before
    # daylist = [7, 5, 5, 5, 6, 6, 8, 12]
    daylist = [7, 5, 6, 6, 6, 6, 8, 12]
    subtrials = 'all'
    
    
# Load time bins for patterns, do all the time for now, can slice later 
seed = 42
total_bins = slice(0, 20)
total_bins_length = total_bins.stop - total_bins.start

In [ ]:
# 2. Load decoding patterns and labels, if not there, create them
pattern_path = os.path.join(result_dir, "decoding_patterns.pickle")
label_path = os.path.join(result_dir, "decoding_labels.pickle")

if os.path.exists(pattern_path):
    with open(pattern_path, "rb") as f:
        patterns = pickle.load(f)
    with open(label_path, "rb") as f:
        labels = pickle.load(f)    
else:
    patterns = {}
    labels = {}
    for ia, animal in enumerate(animal_list):
        day = daylist[ia]
        animal_dir = os.path.join(data_dir, animal, "d"+str(day))
        file_dir = os.path.join(animal_dir, "files")
        
        Fcorr_5hz = np.load(os.path.join(file_dir, "F_5hz.npy"), allow_pickle=True)
        new_im_ts = np.load(os.path.join(file_dir, "timestamps_5hz.npy"), allow_pickle=True)
        
        # # Extract all event time points from new event_df
        num_planes = Fcorr_5hz.shape[0]
        data_loader = DataLoader(animal_dir, num_planes, 0, imaging_system)

        event_df = pd.read_pickle(os.path.join(file_dir, "event_df.pkl"))  # Arduino
        [licks, CS1, CS2, CS3, sucrose, umami] = extract_events(event_df)
        allCS = filter_trials_by_minITI([CS1, CS2, CS3], post_cue_window)
            
        # Normalize signal
        Fcorr_norm_down = normalize_signal(
            Fcorr_5hz, num_planes, "median"
        )  # can be z_score, median, robust_z_score

        # # Extract binned Fcorr around pre and post cue window for each trial, shape is nCS types x ntrials x nCell x nbins, binsize in ms
        Fcorr_around_cue = extract_F_around_events(
            allCS,
            Fcorr_norm_down,
            new_im_ts,
            num_planes,
            pre_cue_window,
            post_cue_window,
            binsize=1000,
            framerate=framerate,
            subtrials=subtrials
        )
        
        X, y = extract_patterns_for_decoding(Fcorr_around_cue, total_bins)
        patterns[animal] = X
        labels[animal] = y        

    with open(pattern_path, "wb") as f:
        pickle.dump(patterns, f)
    with open(label_path, "wb") as f:
        pickle.dump(labels, f)

In [ ]:
# 3. Set decoding parameters
# Choose a decoder, default is linearSVC
clf = LinearSVC()
# clf = SVC(kernel='rbf')
# clf = RandomForestClassifier(n_estimators=100, random_state=seed)
clf_chance = clf        
decoding_pair = ("CS1", "CS2")
testing_pair = decoding_pair
# testing_pair = ("CS2", "CS3")

# set time bins
decoding_time_window = np.arange(0, 20)  # decoding time window
cue_window = np.arange(3,4)
trace_window = np.arange(4,6)
postUS_window_0 = np.arange(6,9)
postUS_window_1 = np.arange(9,12)
# postUS_window_2 = np.arange(12,15)

accuracy = [[] for x in animal_list]
accuracy_chance = [[] for x in animal_list]
subsampling = np.nan
if np.isnan(subsampling):
    niteration = 5
else:
    niteration = 10

In [ ]:
# decided whether decode by clusters
cluster_labels = np.load(os.path.join(result_dir, "clusterlabels.npy")) # each cell's cluster label
populationdata = np.load(os.path.join(result_dir, "populationdata.npy"))
animal_id = np.load(os.path.join(result_dir, "animal_id.npy")) # each cell's animal ID
decode_by_cluster = False
selected_clusters = np.arange(0, 9) # zero indexed, usually from 0 to 8

for cluster in selected_clusters:
    ##--------------------------------------------------------------------------------------------------------
    # 4. Do decoding
    # Get each animal's decoding data with cell indices and cluster ID if needed
    decoded_animals = get_animal_decoding_dict(
        animal_list=animal_list,
        patterns=patterns,
        trial_types=trial_types,
        total_bins_length=total_bins_length,
        decode_by_cluster=decode_by_cluster,
        cluster_labels=cluster_labels,
        selected_clusters=cluster,
        animal_id=animal_id,
    )

    # run decoding across time 
    accuracy, accuracy_chance = do_time_resolved_decoding(
        decoded_animals,
        animal_list,
        decoding_time_window,
        decoding_pair,
        total_bins_length,
        subtrials=subtrials,
        niteration=niteration,
        clf=clf,
        clf_chance=clf_chance,   
        subsampling=subsampling,
        seed=seed,
        testing_pair=testing_pair)

    # accuracy_fixed_time, accuracy_chance_fixed_time = do_time_window_decoding(
    #     decoded_animals,
    #     animal_list,
    #     time_window=,
    #     decoding_pair,
    #     niteration,
    #     clf,
    #     clf_chance,
    #     subsampling,
    #     seed
    # )

    ##--------------------------------------------------------------------------------------------------------
    # 5. Plotting
    fig_accuracy_across_time = plot_decoding_accuracy_across_time(accuracy, accuracy_chance)
    decoder_name = clf.__class__.__name__
    if decode_by_cluster:
        plot_filename = f"{testing_pair[0]}vs{testing_pair[1]}_decoding_{decoder_name}_cluster{cluster+1}.png"
        fig_accuracy_across_time.savefig(os.path.join(result_dir, plot_filename), format="png")
    else:
        plot_filename = f"{testing_pair[0]}vs{testing_pair[1]}_decoding_{decoder_name}.png"    
        fig_accuracy_across_time.savefig(os.path.join(result_dir, plot_filename), format="png")
        break
    plt.close(fig_accuracy_across_time)